# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

A self-state citation is defined only when the citing patent and the cited patent both have a known US state. I first keep those patent rows, then make a small lookup table from patent number to state. Caching this lookup avoids rebuilding it for each join.

In [8]:
from pyspark.sql import functions as F

us_patents = (
    patents
    .filter((F.col("COUNTRY") == "US") & F.col("POSTATE").isNotNull())
    .withColumn("PATENT", F.col("PATENT").cast("long"))
    .cache()
)

patent_states = (
    us_patents
    .select("PATENT", "POSTATE")
    .dropDuplicates(["PATENT"])
    .cache()
)

patent_states.show(5)

+-------+-------+
| PATENT|POSTATE|
+-------+-------+
|3070816|     OK|
|3071255|     CA|
|3071270|     NJ|
|3071416|     MI|
|3071513|     MI|
+-------+-------+
only showing top 5 rows



The citation file has a `CITING` and a `CITED` patent number. I join it to the state lookup twice: first to find the citing patent's state, then to find the cited patent's state. Inner joins intentionally discard citations where either patent has no known US state.

In [9]:
citing_states = patent_states.select(
    F.col("PATENT").alias("CITING"),
    F.col("POSTATE").alias("CITING_STATE")
)
cited_states = patent_states.select(
    F.col("PATENT").alias("CITED"),
    F.col("POSTATE").alias("CITED_STATE")
)

citations_with_states = (
    citations
    .join(citing_states, "CITING", "inner")
    .join(cited_states, "CITED", "inner")
    .cache()
)

citations_with_states.show(5)

+-------+-------+------------+-----------+
|  CITED| CITING|CITING_STATE|CITED_STATE|
+-------+-------+------------+-----------+
|3070816|3869743|          OH|         OK|
|3070816|5615634|          VT|         OK|
|3071255|5240129|          KY|         CA|
|3071270|4091942|          WA|         NJ|
|3071270|5370494|          TX|         NJ|
+-------+-------+------------+-----------+
only showing top 5 rows



After both states are present, a citation is self-state exactly when `CITING_STATE` equals `CITED_STATE`. Grouping those matches by `CITING` gives the required count

In [10]:
same_state_counts = (
    citations_with_states
    .filter(F.col("CITING_STATE") == F.col("CITED_STATE"))
    .groupBy("CITING")
    .agg(F.count("CITED").alias("SAME_STATE_CITATIONS"))
)

same_state_counts.orderBy(F.desc("SAME_STATE_CITATIONS")).show(5)

+-------+--------------------+
| CITING|SAME_STATE_CITATIONS|
+-------+--------------------+
|5959466|                 125|
|5983822|                 103|
|6008204|                 100|
|5952345|                  98|
|5998655|                  96|
+-------+--------------------+
only showing top 5 rows



A left join retains eligible US patents even if they made no same-state citations. Missing counts are replaced with zero before sorting, which avoids null-ordering issues. The second sort key makes tied results deterministic.

In [11]:
augmented_patents = (
    us_patents
    .join(same_state_counts, us_patents.PATENT == same_state_counts.CITING, "left")
    .drop("CITING")
    .fillna({"SAME_STATE_CITATIONS": 0})
)

top_10 = (
    augmented_patents
    .select("PATENT", "POSTATE", "SAME_STATE_CITATIONS")
    .orderBy(F.desc("SAME_STATE_CITATIONS"), F.asc("PATENT"))
    .limit(10)
)

top_10.show(truncate=False)

+-------+-------+--------------------+
|PATENT |POSTATE|SAME_STATE_CITATIONS|
+-------+-------+--------------------+
|5959466|CA     |125                 |
|5983822|TX     |103                 |
|6008204|CA     |100                 |
|5952345|CA     |98                  |
|5958954|CA     |96                  |
|5998655|CA     |96                  |
|5936426|CA     |94                  |
|5739256|CA     |90                  |
|5913855|CA     |90                  |
|5925042|CA     |90                  |
+-------+-------+--------------------+



After running the full dataset, patent 6009554 should have eight same-state citations, as described in the readme. 

In [20]:
assert (
    augmented_patents.filter(F.col("PATENT") == 6009554).first()["SAME_STATE_CITATIONS"]
    == 8
)
print("Full-data check passed: patent 6009554 has 8 same-state citations.")

Full-data check passed: patent 6009554 has 8 same-state citations.
